# Analytics 1 - Firme din Romania (L2)

Notebook de analiza pe datele consolidate L2 (`data.gov.ro/l2_data/`), produse in `data-download-firme-rom.ipynb` din:
- `OD_FIRME` + `OD_CAEN_AUTORIZAT` + `OD_STARE_FIRMA` (Registrul Comertului)
- situatiile financiare `ir` / `uu` / `bl_bs_sl` (data.gov.ro, Guvernul Romaniei)

Fisierele disponibile in `l2_data/`:
- `ir_l2.csv`, `uu_l2.csv`, `bl_bs_sl_l2.csv` - date financiare + identitatea firmei, un rand per CUI
- `caen_autorizat_l2.csv` - coduri CAEN autorizate per firma (neagregat), legat prin `COD_INMATRICULARE`
- `stare_firma_l2.csv` - coduri de stare per firma (neagregat), legat prin `COD_INMATRICULARE`

In [ ]:
import IPython
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 1100)
pd.set_option('display.width', 1000)
# Disable the SettingWithCopyWarning
pd.options.mode.chained_assignment = None

from itables import init_notebook_mode
import itables.options

itables.options.lengthMenu = [5, 10, 15, 25, 50]
init_notebook_mode(all_interactive=True)

import numpy as np
import matplotlib.pyplot as plt

from IPython.display import HTML, display

# Tema inchisa pentru tabelele itables: fundal negru, text alb, headere/sageti de sortare galbene.
# Sortarea pe coloane ramane activa implicit (DataTables) - click pe header pentru asc/desc.
# Acopera atat clasele DataTables 1.x (dataTables_*) cat si cele 2.x (dt-*), pt compatibilitate.
display(HTML("""
<style>
table.dataTable, table.dataTable thead, table.dataTable tbody,
table.dataTable thead th, table.dataTable thead td,
table.dataTable tbody th, table.dataTable tbody td {
    background-color: #000000 !important;
    color: #ffffff !important;
    border-color: #444444 !important;
}

table.dataTable thead th, table.dataTable thead td {
    color: #ffd700 !important;
}

table.dataTable tbody tr:hover td {
    background-color: #222222 !important;
}

.dataTables_wrapper, .dt-container,
.dataTables_wrapper .dataTables_info, .dt-container .dt-info,
.dataTables_wrapper .dataTables_length, .dt-container .dt-length,
.dataTables_wrapper .dataTables_filter, .dt-container .dt-search,
.dataTables_wrapper .dataTables_paginate, .dt-container .dt-paging {
    color: #ffffff !important;
}

.dataTables_wrapper .dataTables_filter input, .dt-container .dt-search input,
.dataTables_wrapper .dataTables_length select, .dt-container .dt-length select,
.dt-input {
    background-color: #000000 !important;
    color: #ffffff !important;
    border: 1px solid #666666 !important;
}

.dataTables_wrapper .dataTables_paginate .paginate_button,
.dt-container .dt-paging .dt-paging-button {
    color: #ffffff !important;
    background: transparent !important;
    border-color: #444444 !important;
}

.dataTables_wrapper .dataTables_paginate .paginate_button.current,
.dt-container .dt-paging .dt-paging-button.current {
    color: #000000 !important;
    background: #ffd700 !important;
    border-color: #ffd700 !important;
}

.dataTables_wrapper .dataTables_paginate .paginate_button.disabled,
.dt-container .dt-paging .dt-paging-button.disabled {
    color: #666666 !important;
}
</style>
"""))

## Incarcare nomenclator CAEN (referinta)

`caen_autorizat_l2.csv` contine doar codurile CAEN brute (nedecodificate - vezi `data-download-firme-rom.ipynb`). Denumirea activitatii se cauta punctual aici, in nomenclatorul din `data.gov.ro/ref_data/`, dupa cheia compusa (`COD_CAEN_AUTORIZAT` normalizat la 4 cifre, `VER_CAEN_AUTORIZAT`) - **nu doar dupa cod**, pentru ca acelasi cod poate insemna activitati diferite in versiuni CAEN diferite (1998/2003/2008/2025).

In [ ]:
from pathlib import Path

REF_DATA_DIR = Path("/Users/tudor/Documents/Data-for-Projects/Cercetare-Research/data.gov.ro/ref_data")

# cheie -> nume fisier in REF_DATA_DIR
REF_FILES = {
    "caen": "N_CAEN.csv",
}

# Cache in memorie, ca sa nu recitim de pe disc daca apelam load_ref de mai multe ori pentru aceeasi cheie.
ref_data = {}


def load_ref(cheie: str, forte_recitire: bool = False, **read_csv_kwargs) -> pd.DataFrame:
    """Incarca (si cacheaza) un fisier din REF_DATA_DIR intr-un DataFrame. Nu face nimic pana nu e apelata explicit."""
    if cheie not in REF_FILES:
        raise KeyError(f"Cheie necunoscuta '{cheie}'. Chei disponibile: {list(REF_FILES)}")
    if forte_recitire or cheie not in ref_data:
        path = REF_DATA_DIR / REF_FILES[cheie]
        # fisierele de referinta sunt delimitate prin '^' (ca fisierele OD_* de sursa), nu prin ','
        ref_data[cheie] = pd.read_csv(path, sep="^", encoding="utf-8-sig", dtype=str, **read_csv_kwargs)
    return ref_data[cheie]


# Incarcam efectiv nomenclatorul CAEN acum (fisier mic, ~3.740 randuri) - il folosim des mai jos.
df_caen = load_ref("caen")
print(f"N_CAEN incarcat: {df_caen.shape[0]} randuri, {df_caen.shape[1]} coloane.")
df_caen.head()

## Incarcare date L2

Celula de mai jos doar **pregateste** caile si un loader generic - nu citeste niciun fisier inca. Incarca punctual, cand ai nevoie, cu `load_l2("cheie")` (sau `load_l2("cheie", nrows=...)` pentru un esantion rapid din fisierele mari).

In [ ]:
from pathlib import Path

L2_DIR = Path("/Users/tudor/Documents/Data-for-Projects/Cercetare-Research/data.gov.ro/l2_data")

# cheie -> nume fisier in L2_DIR
L2_FILES = {
    "ir": "ir_l2.csv",
    "uu": "uu_l2.csv",
    "bl_bs_sl": "bl_bs_sl_l2.csv",
    "caen_autorizat": "caen_autorizat_l2.csv",
    "stare_firma": "stare_firma_l2.csv",
}

# Cache in memorie, ca sa nu recitim de pe disc daca apelam load_l2 de mai multe ori pentru aceeasi cheie.
l2_data = {}


def load_l2(cheie: str, forte_recitire: bool = False, **read_csv_kwargs) -> pd.DataFrame:
    """Incarca (si cacheaza) un fisier din L2_DIR intr-un DataFrame. Nu face nimic pana nu e apelata explicit."""
    if cheie not in L2_FILES:
        raise KeyError(f"Cheie necunoscuta '{cheie}'. Chei disponibile: {list(L2_FILES)}")
    if forte_recitire or cheie not in l2_data:
        path = L2_DIR / L2_FILES[cheie]
        l2_data[cheie] = pd.read_csv(path, encoding="utf-8-sig", **read_csv_kwargs)
    return l2_data[cheie]


print("Fisiere L2 disponibile:", list(L2_FILES))
print("Niciun fisier incarcat inca - foloseste load_l2(cheie) cand ai nevoie de un anumit tabel.")

## Top 50 coduri CAEN in bl_bs_sl (dupa numarul de firme)

Coloana `caen` din `bl_bs_sl_l2.csv` e codul CAEN declarat de firma in situatia financiara — un singur cod per firma, **fara** versiune CAEN asociata (spre deosebire de `OD_CAEN_AUTORIZAT`, care are `VER_CAEN_AUTORIZAT`). Cand un cod exista in mai multe versiuni ale nomenclatorului cu denumiri diferite, alegem denumirea din cea mai recenta versiune disponibila — o aproximare, nu o certitudine, documentata aici ca atare.

In [ ]:
df_bl_bs_sl = load_l2("bl_bs_sl")

# Numarul de firme (randuri) per cod CAEN, top 50. Normalizam codul la 4 cifre
# (cateva coduri din sursa lipsesc zero-ul de inceput, ex. "154" in loc de "0154").
top_50_caen = (
    df_bl_bs_sl["caen"].astype(str).str.strip().str.zfill(4)
    .value_counts()
    .head(50)
    .rename_axis("cod_caen")
    .reset_index(name="numar_firme")
)

# Denumirea CAEN, preferand cea mai recenta versiune a nomenclatorului pentru fiecare cod.
df_caen = load_ref("caen")
coduri_caen = df_caen.assign(CLASA=df_caen["CLASA"].str.strip())
coduri_caen = coduri_caen[coduri_caen["CLASA"].str.fullmatch(r"\d{4}", na=False)]
denumiri_caen = (
    coduri_caen.sort_values("VERSIUNE_CAEN", ascending=False)
    .drop_duplicates("CLASA")
    .set_index("CLASA")["DENUMIRE"]
)

top_50_caen["denumire_activitate"] = top_50_caen["cod_caen"].map(denumiri_caen)

display(top_50_caen)

## Subset: doar firmele din top 50 coduri CAEN

Selectam din `df_bl_bs_sl` doar randurile al caror cod CAEN (normalizat la 4 cifre) e in `top_50_caen`, intr-un `df_top50_caen` nou.

In [ ]:
# Acelasi cod de normalizare ca la calculul top_50_caen (caen e citit ca numar, deci pierde zero-ul de inceput).
df_bl_bs_sl["cod_caen"] = df_bl_bs_sl["caen"].astype(str).str.zfill(4)

df_top50_caen = df_bl_bs_sl[df_bl_bs_sl["cod_caen"].isin(top_50_caen["cod_caen"])].copy()

print(
    f"df_top50_caen: {df_top50_caen.shape[0]} randuri din {df_bl_bs_sl.shape[0]} totale "
    f"({df_top50_caen.shape[0] / df_bl_bs_sl.shape[0] * 100:.1f}%)"
)
df_top50_caen.head()

## Cateva analize pe `df_top50_caen`

- Distributia cifrei de afaceri neta si a numarului mediu de salariati (scara log10, doar valori pozitive - firmele cu 0/negativ sunt excluse din histograme si numarate separat).
- Cele mai frecvente 15 coduri CAEN si judete in acest subset.
- Distributia formelor juridice.
- Relatia cifra de afaceri - numar de salariati (scatter log-log).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Histograma cifrei de afaceri neta (log10, doar valori pozitive)
venituri_pozitive = df_top50_caen.loc[df_top50_caen["cifra_de_afaceri_neta"] > 0, "cifra_de_afaceri_neta"]
excluse_venituri = len(df_top50_caen) - len(venituri_pozitive)
axes[0, 0].hist(np.log10(venituri_pozitive), bins=50, color="steelblue")
axes[0, 0].set_title(f"Cifra de afaceri neta (log10)\n{excluse_venituri} firme cu valoare <=0 excluse")
axes[0, 0].set_xlabel("log10(cifra_de_afaceri_neta)")
axes[0, 0].set_ylabel("numar firme")

# 2. Histograma numarului mediu de salariati (log10, doar valori pozitive)
salariati_pozitivi = df_top50_caen.loc[df_top50_caen["numar_mediu_de_salariati"] > 0, "numar_mediu_de_salariati"]
excluse_salariati = len(df_top50_caen) - len(salariati_pozitivi)
axes[0, 1].hist(np.log10(salariati_pozitivi), bins=40, color="darkorange")
axes[0, 1].set_title(f"Numar mediu de salariati (log10)\n{excluse_salariati} firme cu 0/lipsa excluse")
axes[0, 1].set_xlabel("log10(numar_mediu_de_salariati)")
axes[0, 1].set_ylabel("numar firme")

# 3. Top 15 coduri CAEN dupa numar de firme
top15_caen = top_50_caen.head(15).sort_values("numar_firme")
axes[0, 2].barh(top15_caen["cod_caen"], top15_caen["numar_firme"], color="seagreen")
axes[0, 2].set_title("Top 15 coduri CAEN")
axes[0, 2].set_xlabel("numar firme")

# 4. Top 15 judete
top_judete = df_top50_caen["ADR_JUDET"].value_counts().head(15).sort_values()
axes[1, 0].barh(top_judete.index, top_judete.values, color="indianred")
axes[1, 0].set_title("Top 15 judete")
axes[1, 0].set_xlabel("numar firme")

# 5. Distributie forma juridica (top 6)
top_forme = df_top50_caen["FORMA_JURIDICA"].value_counts().head(6)
axes[1, 1].bar(top_forme.index, top_forme.values, color="mediumpurple")
axes[1, 1].set_title("Forma juridica (top 6)")
axes[1, 1].set_ylabel("numar firme")

# 6. Cifra de afaceri vs. numar de salariati (log-log), doar valori pozitive pe ambele axe
subset_scatter = df_top50_caen[
    (df_top50_caen["cifra_de_afaceri_neta"] > 0) & (df_top50_caen["numar_mediu_de_salariati"] > 0)
]
axes[1, 2].scatter(
    np.log10(subset_scatter["numar_mediu_de_salariati"]),
    np.log10(subset_scatter["cifra_de_afaceri_neta"]),
    s=5, alpha=0.3, color="teal",
)
axes[1, 2].set_title("Cifra de afaceri vs. nr. salariati (log-log)")
axes[1, 2].set_xlabel("log10(numar_mediu_de_salariati)")
axes[1, 2].set_ylabel("log10(cifra_de_afaceri_neta)")

plt.tight_layout()
plt.show()

## Detaliu per judet: histograme, bar chart si scatter pentru fiecare din top 15 judete

Pentru fiecare din cele 15 judete cu cele mai multe firme (in `df_top50_caen`), repetam: histograma cifrei de afaceri, histograma numarului de salariati, top 10 coduri CAEN *specifice judetului* (poate diferi de top-ul global) si scatter-ul cifra de afaceri vs. salariati.

In [ ]:
top_15_judete = df_top50_caen["ADR_JUDET"].value_counts().head(15).index.tolist()

for judet in top_15_judete:
    df_judet = df_top50_caen[df_top50_caen["ADR_JUDET"] == judet]

    fig, axes = plt.subplots(1, 4, figsize=(22, 4.5))
    fig.suptitle(f"{judet} — {len(df_judet)} firme (subset top 50 CAEN)", fontsize=13)

    venituri_pozitive = df_judet.loc[df_judet["cifra_de_afaceri_neta"] > 0, "cifra_de_afaceri_neta"]
    axes[0].hist(np.log10(venituri_pozitive), bins=30, color="steelblue")
    axes[0].set_title("Cifra de afaceri (log10)")
    axes[0].set_xlabel("log10(cifra_de_afaceri_neta)")
    axes[0].set_ylabel("numar firme")

    salariati_pozitivi = df_judet.loc[df_judet["numar_mediu_de_salariati"] > 0, "numar_mediu_de_salariati"]
    axes[1].hist(np.log10(salariati_pozitivi), bins=25, color="darkorange")
    axes[1].set_title("Numar salariati (log10)")
    axes[1].set_xlabel("log10(numar_mediu_de_salariati)")

    top_caen_judet = df_judet["cod_caen"].value_counts().head(10).sort_values()
    axes[2].barh(top_caen_judet.index, top_caen_judet.values, color="seagreen")
    axes[2].set_title("Top 10 coduri CAEN (judet)")
    axes[2].set_xlabel("numar firme")

    subset_scatter = df_judet[(df_judet["cifra_de_afaceri_neta"] > 0) & (df_judet["numar_mediu_de_salariati"] > 0)]
    axes[3].scatter(
        np.log10(subset_scatter["numar_mediu_de_salariati"]),
        np.log10(subset_scatter["cifra_de_afaceri_neta"]),
        s=8, alpha=0.35, color="teal",
    )
    axes[3].set_title("Cifra afaceri vs. salariati")
    axes[3].set_xlabel("log10(numar_mediu_de_salariati)")
    axes[3].set_ylabel("log10(cifra_de_afaceri_neta)")

    plt.tight_layout()
    plt.show()